# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'Unknown Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here we show the available Record Sets and their associated Fields/Columns. All references use their `@id`.

In [ ]:
# Display all record sets and their @id
print('Available Record Sets:')
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"- RecordSet @id: {getattr(rs, '@id', str(rs))} (name: {getattr(rs, 'name', '-')})")
        record_set_ids.append(getattr(rs, '@id', str(rs)))
else:
    # Try top-level recordSet (for older croissant export)
    raw = metadata.to_json() if hasattr(metadata, 'to_json') else dict(metadata)
    top_rs = raw.get('recordSet') or raw.get('record_sets') or []
    for rs in top_rs:
        _id = rs.get('@id', str(rs))
        name = rs.get('name', '-')
        print(f"- RecordSet @id: {_id} (name: {name})")
        record_set_ids.append(_id)

if not record_set_ids:
    print("No record sets found in the metadata.")
else:
    print()
    print('For each record set, showing the Fields/Columns and their @id:')
    for rs_id in record_set_ids:
        print(f"\nRecord Set {rs_id} Fields/Columns:")
        # Try to extract field info via schema API
        try:
            rs_obj = None
            if hasattr(metadata, 'record_sets'):
                for rs in metadata.record_sets:
                    if getattr(rs, '@id', None) == rs_id:
                        rs_obj = rs
                        break
            else:
                pass  # cannot lookup
            fields = getattr(rs_obj, 'fields', []) if rs_obj else []
            if fields:
                for f in fields:
                    print(f"  - Field @id: {getattr(f, '@id', str(f))} (name: {getattr(f, 'name', '-')})")
            else:
                # Try to fallback to the JSON
                raw = metadata.to_json() if hasattr(metadata, 'to_json') else dict(metadata)
                for rs_meta in raw.get('recordSet', []):
                    if rs_meta.get('@id', None) == rs_id:
                        for f in rs_meta.get('field', []):
                            print(f"  - Field @id: {f.get('@id', str(f))} (name: {f.get('name', '-')})")
                        break
        except Exception as e:
            print(f"  (Error reading fields: {e})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note**: Replace the record set and field `@id` variables below with values printed above for your specific exploration.

In [ ]:
# Example: Use the main data record set
# As of Croissant 1.0, datasets often use a main record set with id like 'cr:RecordSet' or similar.
# If you have multiple record sets, you can adjust record_set_ids.

# For this dataset, by introspection or the Croissant, use 'cr:RecordSet' if unsure
record_set_ids = ['cr:RecordSet']
dataframes = {}

for record_set in record_set_ids:
    print(f"Loading records for Record Set: {record_set}")
    # Loading all records from a record set as list-of-dicts
    records = list(dataset.records(record_set=record_set))
    if len(records) == 0:
        print(f"No records found for Record Set {record_set}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Columns in DataFrame for RecordSet {record_set}:\n", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Below, you can pick a numeric field (e.g., 'phq9_score' or 'gad7_score') and a group field (e.g., 'gender' or 'village') from your dataset columns for demonstration.

All field selection below is by DataFrame column name, which matches fields' `@id` in most Croissant datasets.

In [ ]:
# Choose fields for EDA
record_set_id = 'cr:RecordSet'  # Adjust if different
df = dataframes.get(record_set_id)

if df is not None and not df.empty:
    # Try to automatically pick numeric and group fields
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
    print("Numeric candidate columns:", numeric_candidates)
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        # Fallback for demonstration
        numeric_field = df.columns[0]
        print(f"Fallback field: {numeric_field}")

    # Print unique group candidates
    group_candidates = [col for col in df.columns if df[col].dtype == object or pd.api.types.is_string_dtype(df[col])]
    print("Group candidate columns:", group_candidates)
    group_field = group_candidates[0] if group_candidates else df.columns[0]

    threshold = df[numeric_field].dropna().mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

    # Filtering
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (N={len(filtered_df)}):")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records (")
    print(filtered_df[[numeric_field, norm_col]].head())

    if group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (showing mean {numeric_field}):")
        print(grouped_df.head())
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is an example of plotting the distribution of a numeric field and the group-wise average.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and not df.empty:
    # Histogram of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Grouped bar chart
    grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index()
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("No data loaded for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and inspected metadata and data from the Kilifi County Mental Health Survey dataset via its Croissant schema.
- Previewed available record sets, fields, and sample data.
- Demonstrated basic data filtering, normalization, grouping, and visualization of numeric and categorical fields.

These steps provide a foundation for further exploration and advanced analytics using the [`mlcroissant`](https://github.com/mlcommons/croissant) library with AI-ready datasets.